In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
import pandas as pd

from cnn_surgery.lenses.regressor_lens import get_regressor_lens, mse_mae
from cnn_surgery.utils.evaluate_per_class_accuracy import evaluate_classifier, load_testset_data
from cnn_surgery.utils.load_dataset import load_dataset
from cnn_surgery.utils.reconstruct_network import reconstruct_network

DATASET = "mnist"
METRICS_FILENAME = "metrics_merged.csv"

In [ ]:
def test_network_accuracy(weights: np.ndarray | torch.Tensor, activation_fn):
    """
    Reconstructs a network from weights and activation function, evaluates it on the test set
    BEWARE: when using it in unlearning, the input could be a tensor, so be sure to convert it to numpy array first:
        weights = weights.detach().numpy()

    Returns:
        mean accuracy: float
        per class accuracies: list of floats
    """

    if isinstance(weights, torch.Tensor):
        weights = weights.detach().numpy()

    CNNModel = reconstruct_network(weights, activation_fn)
    # this returns a Keras model (Unterthiner code), we have to compile it first
    CNNModel.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    x_test, y_test = load_testset_data(DATASET)
    overall_acc, class_accs = evaluate_classifier(CNNModel, x_test, y_test)
    return overall_acc, class_accs

Loss functions

In [ ]:
# ----- loss functions -----
simple_loss = lambda pred, target_idx: pred[target_idx]  # minimize the accuracy of the target class

def unlearning_and_faithfulness_loss(pred, true, target_idx):  # needs a better name
    target_term = pred[target_idx]  # we want to minimize this

    # # set target index zero for this term by multiplying with a mask
    # mask = torch.ones_like(pred, requires_grad=False)
    # mask[target_idx] = 0
    # maintain_rest_term = (((true - pred) * mask) ** 2).mean()  # we want to maintain the accuracy of the other classes

    # mse = ((true - pred) ** 2).mean()  # we want the regressorlens to remain faithful to the actual performance
    mae = (true - pred).abs().mean()

    # return target_term + mse
    return target_term + 3 * mae  # minimize predicted target performance

Load dataset and train model

In [ ]:
early = load_dataset('mnist', metrics_file='metrics_merged_mnist_early.csv', load_class_acc=True, stage='early')
middle = load_dataset('mnist', metrics_file='metrics_merged_mnist_middle.csv', load_class_acc=True, stage='middle')
final = load_dataset('mnist', metrics_file='metrics_merged.csv', load_class_acc=True, stage='final')

train_early, test_early, val_early = early
train_middle, test_middle, val_middle = middle
train_final, test_final, val_final = final

weights_train = np.concatenate([train_early[0], train_middle[0], train_final[0]])
weights_val = np.concatenate([val_early[0], val_middle[0], val_final[0]])
weights_test = np.concatenate([test_early[0], test_middle[0], test_final[0]])

accuracies_train = np.concatenate([train_early[1][:, -10:], train_middle[1][:, -10:], train_final[1][:, -10:]])
accuracies_val = np.concatenate([val_early[1][:, -10:], val_middle[1][:, -10:], val_final[1][:, -10:]])
accuracies_test = np.concatenate([test_early[1][:, -10:], test_middle[1][:, -10:], test_final[1][:, -10:]])

configs_train = pd.concat([train_early[2], train_middle[2], train_final[2]], ignore_index=True)
configs_val = pd.concat([val_early[2], val_middle[2], val_final[2]], ignore_index=True)
configs_test = pd.concat([test_early[2], test_middle[2], test_final[2]], ignore_index=True)

# all model indices are the same across training stages
assert all(train_early[2].index.map(lambda x: x.split('/')[-2]).values == train_middle[2].index.map(lambda x: x.split('/')[-2]).values)
assert all(train_early[2].index.map(lambda x: x.split('/')[-2]).values == train_final[2].index.map(lambda x: x.split('/')[-2]).values)

RegressorLens = get_regressor_lens(weights_train, accuracies_train, weights_val, accuracies_val, device='cpu')

In [ ]:
# find the index of the best-performing model in the test set
overall_accuracies_test = np.concatenate([test_early[1][:, 0], test_middle[1][:, 0], test_final[1][:, 0]])
best_model_idx = np.argmax(overall_accuracies_test)
print(f"Best model index in test set: {best_model_idx}, accuracy: {overall_accuracies_test[best_model_idx]}")

MODEL_IDX = best_model_idx  # you can change this to any index you want

In [ ]:
# # ----- Load dataset -----
# train, test, val = load_dataset(DATASET, metrics_file=METRICS_FILENAME, load_class_acc=True)
# weights_train, outputs_train, configs_train = train
# weights_test, outputs_test, configs_test = test
# weights_val, outputs_val, configs_val = val

# train_class_accuracies = outputs_train[:, -10:]
# test_class_accuracies = outputs_test[:, -10:]
# val_class_accuracies = outputs_val[:, -10:]

# # ----- Load and train regressor lens -----
# # Currently the get_regressor_lens function trains a new lens from scratch
# # It would be good just use a good and pretrained one once the tuning has been done
# # In that case we could rewrite get_regressor_lens to accept a parameter that decides whether to load a model or initialize and train a new one
# RegressorLens = get_regressor_lens(weights_train, train_class_accuracies, weights_val, val_class_accuracies, device="cpu")

unlearning

In [ ]:
assert isinstance(RegressorLens, nn.Module), f"RegressorLens is not an instance of torch.nn.Module but {type(RegressorLens)}"
input_weights: np.ndarray = weights_test[MODEL_IDX]
steps: int = 200
step_size: float = 1.0
target_class: int = 5
og_config: pd.Series = configs_test.iloc[MODEL_IDX]

"""
    input_weights: np.ndarray, weights of the model to be unlearned
    steps: int - number of optimization steps
    lens: torch.nn.Module - pretrained regressor lens model
    step_size: float - step size for the optimization
    target_class: int - class index to be unlearned
    og_config - original configuration of the model to be unlearned (for reconstructing the model)
"""

print("Starting unlearning procedure")
diffs_list = []
loss_list = []
target_term_list = []
mse_term_list = []
mae_term_list = []

# convert input weights to tensor for optimization
doctored_input_weights = torch.tensor(input_weights, requires_grad=True, dtype=torch.float32)

RegressorLens.eval()

# optimize only the INPUT weights
for param in RegressorLens.parameters():
    param.requires_grad = False

for i in tqdm(range(steps), desc="Unlearning in progress"):
    pred: torch.Tensor = RegressorLens(doctored_input_weights.unsqueeze(0)).squeeze(0)  # forward pass

    # this is expensive: a new model needs to be reconstructed and evaluated at every step
    true = torch.tensor(
        test_network_accuracy(doctored_input_weights.detach().numpy(), og_config["config.activation"])[1],
        dtype=torch.float32,
    )

    loss = unlearning_and_faithfulness_loss(pred, true, target_class)

    # for analysis
    target_term: float = pred[target_class].item()
    # mse_term: float = ((true - pred) ** 2).mean().item()
    mae_term: float = (true - pred).abs().mean().item()

    loss.backward()  # compute gradients
    gradients = doctored_input_weights.grad

    with torch.no_grad():
        doctored_input_weights -= step_size * gradients  # gradient step # type: ignore
        doctored_input_weights.grad.zero_()  # zero gradients # type: ignore

    mean_diff = abs((pred.detach().numpy() - np.array(true)).mean())

    loss_list.append(loss.item())
    target_term_list.append(target_term)
    # mse_term_list.append(mse_term)
    mae_term_list.append(mae_term)
    diffs_list.append(mean_diff)

evaluation

In [ ]:
# ------- loss plotting -------
plt.figure(figsize=(10, 6))
plt.plot(loss_list, label="Total loss")
plt.plot(target_term_list, label="Target class penalty term")
# plt.plot(mse_term_list, label="MSE (RegressorLens faithfulness) term")
plt.plot(mae_term_list, label="MAE (RegressorLens faithfulness) term")
plt.xlabel("Unlearning step")
plt.ylabel("Loss")
plt.title("Loss over unlearning steps")
plt.ylim(0, max(loss_list) * 1.2)
plt.legend()
plt.show()

assert isinstance(RegressorLens, nn.Module), f"RegressorLens is not an instance of torch.nn.Module but {type(RegressorLens)}"
preds = RegressorLens(doctored_input_weights.unsqueeze(0)).squeeze().detach().numpy()
actual_accs = test_network_accuracy(doctored_input_weights.detach().numpy(), configs_test.iloc[MODEL_IDX]["config.activation"])[1]
before_accs = accuracies_test[MODEL_IDX]

# ------- accuracy plotting -------
classes = np.arange(len(preds))
bar_width = 0.25

plt.figure(figsize=(10, 6))
plt.bar(classes - bar_width, before_accs, width=bar_width, label="Before unlearning", alpha=0.7)
plt.bar(classes, actual_accs, width=bar_width, label="After unlearning", alpha=0.7)
plt.bar(classes + bar_width, preds, width=bar_width, label="Predicted by RegressorLens", alpha=0.7)
plt.xlabel("Class index")
plt.ylabel("Accuracy")
plt.title("Class-wise accuracies: Before Unlearning vs Predicted & Actual, after Unlearning")
plt.xticks(classes)
plt.ylim(0, 1)
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.show()


# ------- error plotting -------
plt.figure(figsize=(10, 6))
plt.plot(diffs_list)
plt.xlabel("Unlearning step")
plt.ylabel("Mean prediction error")
plt.title("Mean prediction error over unlearning steps")
plt.ylim(0, max(diffs_list) * 1.2)
plt.show()